In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_PUBLIC = PROJECT_ROOT / "artifacts" / "public"
DATA_PUBLIC = PROJECT_ROOT / "data" / "public"
MODELS_PUBLIC = PROJECT_ROOT / "models" / "public"


# Day 05.4 / Day 05.4b · Fallback, Flags and Local Policies

**Estado actual**: el run histórico `20260310T093320Z` sigue invalidado y fuera de superficie activa. El rerun limpio `20260310T110104Z` cierra Day 05.4 con `keep_current_operational_policy`, y la microiteración `Day 05.4b` auditada en `20260310T132832Z` vuelve a cerrar con `keep_current_operational_policy` por regla de stop.

Este notebook sigue siendo el notebook previo a Brent: separa evidencia histórica inválida, evidencia válida de Day 05.4 y cierre auditado de Day 05.4b, usando solo código y artefactos reales del repo.



## 1. Cadena forense del bloque

La lectura correcta del bloque ya no es una sola corrida:
1. `20260310T093320Z`: run inválido por `scoring_contract_mismatch`.
2. `20260310T110104Z`: rerun limpio Day 05.4, sin promoción.
3. `20260310T132832Z`: Day 05.4b residual/composite, auditado y también sin promoción.

A partir de aquí, la evidencia válida para decidir antes de Brent sale solo de `20260310T110104Z` y `20260310T132832Z`.


In [2]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
PROJECT_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == "notebooks" else PROJECT_ROOT
INVALIDATED_RUN_ID = "20260310T093320Z"
DAY054_VALID_RUN_ID = "20260310T110104Z"
DAY054B_RUN_ID = "20260310T132832Z"

invalidated_path = PROJECT_ROOT / "artifacts" / "public" / "metrics" / "day05_4" / "invalidations" / f"{INVALIDATED_RUN_ID}_invalidated.json"
day054_summary_path = PROJECT_ROOT / "artifacts" / "public" / "metrics" / "day05_4" / f"{DAY054_VALID_RUN_ID}_run_summary.json"
day054b_summary_path = PROJECT_ROOT / "artifacts" / "public" / "metrics" / "day05_4" / f"{DAY054B_RUN_ID}_run_summary.json"
day054b_trials_path = PROJECT_ROOT / "artifacts" / "public" / "metrics" / "day05_4" / f"{DAY054B_RUN_ID}_policy_trials.csv"
registry_csv_path = PROJECT_ROOT / "artifacts" / "public" / "metrics" / "final_baseline_vs_candidates.csv"

invalidated_payload = json.loads(invalidated_path.read_text(encoding="utf-8"))
day054_summary = json.loads(day054_summary_path.read_text(encoding="utf-8"))
day054b_summary = json.loads(day054b_summary_path.read_text(encoding="utf-8"))
day054b_trials = pd.read_csv(day054b_trials_path)
registry_tail = pd.read_csv(registry_csv_path).tail(4)

print("invalidated_run_id:", invalidated_payload["invalidated_run_id"])
print("day054_valid_run_id:", day054_summary["run_id"], "->", day054_summary["decision"])
print("day054b_run_id:", day054b_summary["run_id"], "->", day054b_summary["decision"])
print("winning_policy_day054b:", day054b_summary["winning_policy"])


invalidated_run_id: 20260310T093320Z
day054_valid_run_id: 20260310T110104Z -> keep_current_operational_policy
day054b_run_id: 20260310T132832Z -> keep_current_operational_policy
winning_policy_day054b: None



## 2. Base válida Day 05.4 reutilizada por Day 05.4b

Day 05.4b no rehace la reparación: la toma como contrato base. Eso significa que sigue usando el scorer contract-aware, el preflight exacto contra `metadata.json` y registry, y la cuarentena del run inválido.

La microiteración 05.4b solo añade reglas más precisas sobre `PRODUCT_003 residual` y una compuesta final con las ganancias ya conocidas de `PRODUCT_005` y `PRODUCT_003 dominante`.


In [3]:
preflight_rows = []
for reference_name, payload in day054b_summary["preflight"].items():
    for metric_name, metric_payload in payload["metadata_check"].items():
        preflight_rows.append({
            "reference_name": reference_name,
            "comparison": "metadata.json",
            "metric": metric_name,
            "actual": metric_payload["actual"],
            "expected": metric_payload["expected"],
            "abs_delta": metric_payload["abs_delta"],
            "ok": metric_payload["ok"],
        })
    for metric_name, metric_payload in payload["registry_check"].items():
        preflight_rows.append({
            "reference_name": reference_name,
            "comparison": "registry_row",
            "metric": metric_name,
            "actual": metric_payload["actual"],
            "expected": metric_payload["expected"],
            "abs_delta": metric_payload["abs_delta"],
            "ok": metric_payload["ok"],
        })
preflight_df = pd.DataFrame(preflight_rows)
print(preflight_df.to_string(index=False))


              reference_name    comparison            metric   actual  expected  abs_delta   ok
               champion_pure metadata.json          accuracy 0.900681  0.900681        0.0 True
               champion_pure metadata.json balanced_accuracy 0.881431  0.881431        0.0 True
               champion_pure metadata.json          top1_hit 0.772503  0.772503        0.0 True
               champion_pure metadata.json          top2_hit 0.882643  0.882643        0.0 True
               champion_pure metadata.json          coverage 1.000000  1.000000        0.0 True
               champion_pure  registry_row          accuracy 0.900681  0.900681        0.0 True
               champion_pure  registry_row balanced_accuracy 0.881431  0.881431        0.0 True
               champion_pure  registry_row          top1_hit 0.772503  0.772503        0.0 True
               champion_pure  registry_row          top2_hit 0.882643  0.882643        0.0 True
               champion_pure  registry_r


## 3. Day 05.4b · Radar residual y smokes previos al rerun

Day 05.4b no explora familias nuevas: intenta rescatar `PRODUCT_003 residual` con tres cortes operativos observables y luego componerlo con `PRODUCT_005` y `PRODUCT_003 dominante`.

Los smokes previos al rerun se anclan al audit limpio `20260310T110104Z`. Si ahí ya aparece divergencia frente a la lectura fuerte heredada de Notebook 19, esa divergencia debe quedar explícita y no esconderse detrás del rerun.


In [4]:
smoke_df = pd.DataFrame(day054b_summary["pre_rerun_smokes"])
trial_focus_columns = [
    "policy_variant",
    "selected_events",
    "overrides_count",
    "overrides_harmed",
    "top1_hit",
    "top2_hit",
    "delta_top1_vs_operational",
    "delta_top2_vs_operational",
    "decision_label",
    "failure_reason",
]
radar_df = smoke_df.merge(day054b_trials[trial_focus_columns], on="policy_variant", how="left")
print(radar_df.to_string(index=False))
print("\nDiferencias frente a Notebook 19 / lectura fuerte esperada:")
for line in day054b_summary["notebook19_differences"]:
    print("-", line)


                                   policy_variant  selected_events_x  improves_top1_vs_champion  harms_top1_vs_champion  improves_top2_vs_champion  harms_top2_vs_champion  expected_selected_events  expected_top1_improves  max_top1_harms  alignment_ok                                                                                                                                      notebook19_alignment_note  selected_events_y  overrides_count  overrides_harmed  top1_hit  top2_hit  delta_top1_vs_operational  delta_top2_vs_operational  decision_label  failure_reason
      day054b_go_b_residual_SUPPLIER_050_clean_v1                 26                         19                       7                          0                       0                        19                      19               0         False Operationaliza el subconjunto limpio de PRODUCT_003 residual donde el champion cae en SUPPLIER_050 y baseline deja fallback defendible sin harmed detectables.                 26  


## 4. Candidata compuesta y regla de stop

La compuesta final `day054b_composite_go_c_plus_go_b_dominant_plus_residual_precision_v1` reúne:
- `PRODUCT_005` como fallback top-1 defendible;
- `PRODUCT_003 dominante` como mejora de `Top-2`/ranking dentro de una policy mayoritariamente sustentada por overrides reales de `PRODUCT_005` y residual;
- `PRODUCT_003 residual precision` como intento de limpiar el residual antes de Brent.

El bloque se cierra si esta compuesta no supera a la policy operativa vigente bajo el gate actual.


In [5]:
composite_row = day054b_trials.loc[
    day054b_trials["policy_variant"].eq("day054b_composite_go_c_plus_go_b_dominant_plus_residual_precision_v1")
].copy()
composite_columns = [
    "policy_variant",
    "selected_events",
    "overrides_count",
    "overrides_improved",
    "overrides_harmed",
    "top1_hit",
    "top2_hit",
    "delta_top1_vs_champion",
    "delta_top2_vs_champion",
    "delta_top1_vs_operational",
    "delta_top2_vs_operational",
    "material_improvement_ok",
    "decision_label",
    "failure_reason",
]
print(composite_row[composite_columns].to_string(index=False))


                                                      policy_variant  selected_events  overrides_count  overrides_improved  overrides_harmed  top1_hit  top2_hit  delta_top1_vs_champion  delta_top2_vs_champion  delta_top1_vs_operational  delta_top2_vs_operational  material_improvement_ok decision_label          failure_reason
day054b_composite_go_c_plus_go_b_dominant_plus_residual_precision_v1              464               64                  57                 7  0.791493  0.890619                 0.01899                0.007976                   0.001899                   0.030384                     True         reject champion_guardrail_fail



## 5. Cierre auditado antes de Brent

La lectura final del bloque queda así:
- Day 05.4 sigue cerrado y válido con `keep_current_operational_policy`.
- Day 05.4b también cierra con `keep_current_operational_policy`, porque la compuesta mejora localmente pero rompe el guardrail por `overrides_harmed = 7`.
- No se reabre la gobernanza del gate dentro de Day 05.4b.
- No se abre ninguna subiteración `Day 05.4x` adicional antes de decidir si el siguiente foco será Brent o producto/despliegue/presentación.


In [6]:
decision_df = pd.DataFrame([
    {
        "invalidated_run_id": invalidated_payload["invalidated_run_id"],
        "day054_valid_run_id": day054_summary["run_id"],
        "day054_decision": day054_summary["decision"],
        "day054b_run_id": day054b_summary["run_id"],
        "day054b_decision": day054b_summary["decision"],
        "current_operational_policy_variant": day054b_summary["current_operational_policy_variant"],
        "brent_opened": "no",
    }
])
print(decision_df.to_string(index=False))
print("\nRegistry tail:")
print(registry_tail[["run_id", "day_id", "model_variant", "promotion_decision"]].to_string(index=False))


invalidated_run_id day054_valid_run_id                 day054_decision   day054b_run_id                day054b_decision                                                                     current_operational_policy_variant brent_opened
  20260310T093320Z    20260310T110104Z keep_current_operational_policy 20260310T132832Z keep_current_operational_policy V2_TRANSPORT_ONLY_LR_smote_0.5_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLLOW_PRODUCT_003_SUPPLIER_009_v1           no

Registry tail:
          run_id    day_id                                          model_variant              promotion_decision
20260310T093320Z  Day 05.4           day054_go_b_dominant_fallback_to_baseline_v1                         promote
20260310T110104Z  Day 05.4  day054_run_closure_keep_current_operational_policy_v1 keep_current_operational_policy
20260310T132832Z Day 05.4b day054b_run_closure_keep_current_operational_policy_v1 keep_current_operational_policy
20260310T140709Z Day 05.4c day054c_run_closure_keep_curr


## 6. Day 05.4c · Endurecimiento final de `SUPPLIER_050 clean`

Tras el cierre auditado de Day 05.4b se autorizó una única microiteración final `Day 05.4c` para intentar eliminar los `7 harmed` concentrados en `SUPPLIER_050 clean` sin reabrir el gate ni crear un notebook nuevo.

La regla canónica elegida fue endurecer ese sub-slice con `v41_transport_rank_event <= 2`, manteniendo intacto el resto del bloque:
- `PRODUCT_005` como fallback local defendible;
- `PRODUCT_003 dominante` como mejora de `Top-2`/ranking dentro de la compuesta;
- `PRODUCT_003 residual` reducido a tres cortes: `SUPPLIER_050_rank2_clean`, `SUPPLIER_019_low_conf` y `outer_terminal_extension`.

El objetivo ya no era solo ganar en agregado, sino ganar sin `harmed` y seguir superando a la mejor policy operativa vigente cerrada en gobernanza.


In [7]:
DAY054C_RUN_ID = "20260310T140709Z"
day054c_summary_path = PROJECT_ROOT / "artifacts" / "public" / "metrics" / "day05_4" / f"{DAY054C_RUN_ID}_run_summary.json"
day054c_trials_path = PROJECT_ROOT / "artifacts" / "public" / "metrics" / "day05_4" / f"{DAY054C_RUN_ID}_policy_trials.csv"
day054c_summary = json.loads(day054c_summary_path.read_text(encoding="utf-8"))
day054c_trials = pd.read_csv(day054c_trials_path)
day054c_focus = {
    "run_id": day054c_summary["run_id"],
    "decision": day054c_summary["decision"],
    "base_day054b_valid_run_id": day054c_summary["base_day054b_valid_run_id"],
    "current_operational_policy_variant": day054c_summary["current_operational_policy_variant"],
}
print(pd.Series(day054c_focus).to_string())


run_id                                                                 20260310T140709Z
decision                                                keep_current_operational_policy
base_day054b_valid_run_id                                              20260310T132832Z
current_operational_policy_variant    V2_TRANSPORT_ONLY_LR_smote_0.5_WITH_DETERMINIS...



## 7. Smokes Day 05.4c, compuesta final y cierre definitivo antes de Brent

El radar Day 05.4c debía demostrar tres cosas a la vez:
1. que el endurecimiento `SUPPLIER_050_rank2_clean` separaba de verdad los `19 clean wins` de los `7 harmed` observados en Day 05.4b;
2. que los cortes ya limpios (`SUPPLIER_019_low_conf` y `outer_terminal_extension`) seguían reproduciéndose;
3. que la compuesta final mantenía las ganancias agregadas y además cerraba con `overrides_harmed = 0`.

La lectura final del rerun auditado es la que decide el bloque:
- si el endurecimiento no reproduce el sub-slice esperado, debe quedar explícito;
- si la compuesta elimina el daño pero no supera a la policy operativa vigente en `Top-1`, el bloque Day 05.4x se cierra igualmente sin promoción.


In [8]:
day054c_smoke_df = pd.DataFrame(day054c_summary["pre_rerun_smokes"])
day054c_radar_df = day054c_smoke_df.merge(day054c_trials[trial_focus_columns], on="policy_variant", how="left")
print(day054c_radar_df.to_string(index=False))
print("\nDiferencias frente a Notebook 19 / Day 05.4b:")
for line in day054c_summary["notebook19_differences"]:
    print("-", line)

print("\nCompuesta Day 05.4c:")
day054c_composite_row = day054c_trials.loc[
    day054c_trials["policy_variant"].eq("day054c_composite_go_c_plus_go_b_dominant_plus_residual_rank2_precision_v1")
].copy()
print(day054c_composite_row[composite_columns].to_string(index=False))

day054c_decision_df = pd.DataFrame([
    {
        "invalidated_run_id": invalidated_payload["invalidated_run_id"],
        "day054_valid_run_id": day054_summary["run_id"],
        "day054_decision": day054_summary["decision"],
        "day054b_run_id": day054b_summary["run_id"],
        "day054b_decision": day054b_summary["decision"],
        "day054c_run_id": day054c_summary["run_id"],
        "day054c_decision": day054c_summary["decision"],
        "brent_opened": "no",
    }
])
print("\nCadena final del bloque:")
print(day054c_decision_df.to_string(index=False))
print("\nRegistry tail actualizado:")
print(pd.read_csv(registry_csv_path).tail(5)[["run_id", "day_id", "model_variant", "promotion_decision"]].to_string(index=False))


                                   policy_variant  selected_events_x  improves_top1_vs_champion  harms_top1_vs_champion  improves_top2_vs_champion  harms_top2_vs_champion  expected_selected_events  expected_top1_improves  max_top1_harms  alignment_ok                                                                                                                                 notebook19_alignment_note  selected_events_y  overrides_count  overrides_harmed  top1_hit  top2_hit  delta_top1_vs_operational  delta_top2_vs_operational  decision_label  failure_reason
day054c_go_b_residual_SUPPLIER_050_rank2_clean_v1                  0                          0                       0                          0                       0                        19                      19               0         False Endurece el residual SUPPLIER_050 con rank transport <= 2 para quedarse solo con el subconjunto limpio que preserva las ganancias sin harmed detectables.                  0            